# Performing Domain Analysis with AI  
## Leveraga SA Model, Write Up, and Results on Strategy to develop a first cut Architecture

First we load up the PEAK Model

### Model-Specific Code (Do Not Modify)

This section contains code that is specific to the system model. It is updated only when the model is changed and should not require user modifications under normal circumstances.

If a new model is introduced, ensure this section is reviewed and updated as needed.


In [1]:
#!pip install --upgrade git+https://github.com/tkSDISW/Capella_Tools 
import capellambse.decl

from capella_tools import capellambse_helper

from IPython import display as diag_display
resources = {
    "PEAK": "PEAK/PEAK",
}
path_to_model = "../PEAK.aird"
model = capellambse.MelodyModel(path_to_model, resources=resources)
from capella_tools import Pub4C
# Instantiate the class with the traceability file
traceability_store = Pub4C.Traceability_Store("../PEAK/traceability")


## 🔄 Embedding Generation Process

Next we load evaluate whether we ne to update the embedings. If there older than model we will update thems. 🚀


In [2]:

from capella_tools  import capella_embeddings_manager

# Generate embeddings for all objects
model_embedding_manager = capella_embeddings_manager.EmbeddingManager()

embedding_file = "embeddings.json" 
model_embedding_manager.set_files( path_to_model , embedding_file)

model_embedding_manager.create_model_embeddings(model)

✅ EmbeddingManager initialized
🔐 API Key: Loaded from secrets
🌐 Base URL: https://api.openai.com/v1
🤖 Model: gpt-4o
Loading embeddings
embeddings loaded.


## 🎯 Define the model element or diagram for analysis.

We use the embedding to locate a diagram that will be used for basis of analysis. It a OCB diagrams with relations to all the functional chains. 

> 💡 **Tip:** If you're unsure about the model structure, review the documentation or refer to the model's diagrams for additional guidance.


In [3]:

selected_objects = model_embedding_manager.query_and_select_top_objects("[LAB] Water Generation - Tank QAS and Params", top_n=1)



This is a list of ranked Objects Based on Query:
Index: 0, Name: [LAB] Water Generation - Tank  QAS and Params, Similarity: 0.90, Type: Diagram, Phase: Logical Architecture LA, Source: , Target: 


## 📝 Generate Structured Input File 

We then generate the structured input file for all the matching objects and there related objects.
A file "capella_model.yaml" is written it you want to look at it. 

In [4]:
#Workflow
from capella_tools import capellambse_yaml_manager
#import capellambse_yaml_manager
yaml_handler = capellambse_yaml_manager.CapellaYAMLHandler()
   
#Generate YAML for the logical component and append to the file
for object in  selected_objects :  
    yaml_handler.generate_yaml(model.by_uuid(object["uuid"]))  

yaml_handler.generate_traceability_related_objects(model,traceability_store)

#yaml_handler.display()
yaml_handler.generate_yaml_referenced_objects()
#yaml_handler.display()

yaml_handler.write_output_file()


## ⚙️ Execute  Prompt


Execute the prompt that will use the model and system document. 

In [5]:
import os
from openai import OpenAI
from IPython.core.display import HTML
from IPython.display import display, clear_output, Markdown, IFrame
from capella_tools  import Open_AI_RAG_manager


#print(object)
# Step 1: Get YAML content
yaml_content = yaml_handler.get_yaml_content()

# Step 2: Invoke ChatGPT for analysis
analyzer = Open_AI_RAG_manager.ChatGPTAnalyzer(yaml_content)#analyzer.add_text_file_to_messages("PEAK System.docx")
prompt = """
Write a report on from this diagram. 
Specifically call attention to how the PEAK water subsystem  has been decomposed.
-Detail the specific related requirements to elements of the diagram,
-Detail the specific related property values to elements of the diagram,
"""
analyzer.follow_up_prompt(prompt)
chatgpt_response = analyzer.get_response()

✅ ChatGPTAnalyzer initialized
🔐 API Key: Loaded from secrets
🌐 Base URL: https://api.openai.com/v1
🤖 Model: gpt-4o


**Your prompt:** 
Write a report on from this diagram. 
Specifically call attention to how the PEAK water subsystem  has been decomposed.
-Detail the specific related requirements to elements of the diagram,
-Detail the specific related property values to elements of the diagram,


**Response:**

# Report on the PEAK Water Subsystem Decomposition

## Introduction
The PEAK water subsystem is a critical component of the overall system model, designed to ensure the provision of potable water in crisis situations. This report details the decomposition of the PEAK water subsystem, highlighting its components, related requirements, and property values.

## Decomposition of the PEAK Water Subsystem

### Components of the Water Subsystem
The PEAK water subsystem is composed of several logical components, each serving a specific function in the water purification and distribution process. The key components include:

1. **Water Purification**: This component is responsible for purifying water to make it safe for consumption. It includes filtration, distribution capability, and a storage container. The purification process is capable of handling fresh, brackish, or salt water.

2. **Water Filtration**: Ensures that water is safe for drinking and hygiene. It is part of the purification process and adheres to water safety standards.

3. **Storage Container**: Provides a means to hold and distribute purified water. It must meet durability and lightweight standards, as specified in logistical standards for emergency supplies.

4. **Controls System**: Manages the operation of the water subsystem, including sensing and pumping water.

### Logical Functions
The water subsystem includes several logical functions that facilitate the purification and distribution of water:

- **Filter Water**: Pumps and filters water, pulling it from the environment.
- **Purify Water**: Removes bacteria from water, ensuring it is safe for consumption.
- **Store the Water**: Stores purified water in a vessel for distribution.
- **Sense and Pump Water**: Manages the sensing and pumping of water to ensure efficient operation.

### Functional Exchanges
The water subsystem involves several functional exchanges that facilitate the flow of water and control signals between components:

- **Filtered Water**: The exchange between the Filter Water and Purify Water functions.
- **Purified Water**: The exchange between the Purify Water and Store the Water functions.
- **Control Pump**: The exchange between the Sense and Pump Water and Filter Water functions.

## Related Requirements

### Include Storage Container
- **Requirement**: The water system shall include a storage container.
- **Rationale**: To store the purified water.

### Size
- **Requirement**: The storage container shall be capable of containing at least X liters of water.

## Related Property Values

### Container Size
- **Units**: Liter
- **Value**: 0.0 (Placeholder for actual value)

### Water Consumer
- **Units**: Quantity
- **Value**: Integer values associated with different water consumers.

### Water Consumption Rate
- **Units**: Liter/Day
- **Value**: 3.7

## Conclusion
The PEAK water subsystem is a well-structured component of the overall system, designed to ensure the provision of safe and potable water in crisis situations. Its decomposition into logical components and functions, along with the associated requirements and property values, highlights its critical role in the system's operation. The subsystem's design ensures compliance with safety standards and logistical requirements, making it a reliable solution for emergency water provision.

**Token Usage Info:**

Tokens used: prompt=48508, completion=649, total=49157

## 💬 Launch Interactive Chat on Structured Input




In [6]:
print("Done")

Done


# 